In [ ]:
import pandas as pd


datasets = {
    "releases": {
        "2025": "https://data.texas.gov/resource/q4fw-9sy9.csv?$limit=60000",
        "2024": "https://data.texas.gov/resource/5n6q-4r86.csv?$limit=60000",
        "2023": "https://data.texas.gov/resource/xad4-d2x2.csv?$limit=60000"
    },
    "receives": {
        "2025": "https://data.texas.gov/resource/8ha4-kikr.csv?$limit=60000",
        "2024": "https://data.texas.gov/resource/jrcd-cwun.csv?$limit=60000"
    }
}

def fetch_and_normalize(category_dict):
    
    combined_dfs = []
    for year, url in category_dict.items():
        print(f"Fetching {year} data...")
        df = pd.read_csv(url)
        
        df.columns = df.columns.str.replace('_', '').str.lower()
        
        df['data_cohort_year'] = int(year)
        
        combined_dfs.append(df)
        
    return pd.concat(combined_dfs, ignore_index=True)


print("--- FETCHING RELEASES ---")
df_releases = fetch_and_normalize(datasets["releases"])

print("\n--- FETCHING RECEIVES ---")
df_receives = fetch_and_normalize(datasets["receives"])


print("\n--- INGESTION COMPLETE ---")
print(f"Total Combined Releases Rows: {df_releases.shape[0]} | Columns: {list(df_releases.columns)}")
print(f"Total Combined Receives Rows: {df_receives.shape[0]} | Columns: {list(df_receives.columns)}")

--- FETCHING RELEASES ---
Fetching 2025 data...
Fetching 2024 data...
Fetching 2023 data...

--- FETCHING RECEIVES ---
Fetching 2025 data...
Fetching 2024 data...

--- INGESTION COMPLETE ---
Total Combined Releases Rows: 142307 | Columns: ['releasedate', 'releasetype', 'inmatetype', 'gender', 'race', 'age', 'county', 'offensecode', 'offense', 'offensedescription', 'sentencedate', 'offensedate', 'sentenceyears', 'data_cohort_year']
Total Combined Receives Rows: 108739 | Columns: ['receivedate', 'receivetype', 'inmatetype', 'gender', 'race', 'age', 'county', 'offensecode', 'offense', 'offensedescription', 'sentencedate', 'offensedate', 'sentenceyears', 'data_cohort_year']


In [ ]:
import pandas as pd
import pandas_gbq


PROJECT_ID = "texas-recidivism-rates" 
DATASET_ID = "texas_criminal_justice"

print("Uploading releases data to BigQuery...")

pandas_gbq.to_gbq(
    df_releases, 
    destination_table=f"{DATASET_ID}.stg_raw_releases", 
    project_id=PROJECT_ID, 
    if_exists='replace'
)

print("Uploading receives data to BigQuery...")
pandas_gbq.to_gbq(
    df_receives, 
    destination_table=f"{DATASET_ID}.stg_raw_receives", 
    project_id=PROJECT_ID, 
    if_exists='replace'
)

print("Raw staging tables successfully pushed to BigQuery!")

Uploading releases data to BigQuery...


c:\Users\vicly\OneDrive\Documents\texas_recidivism_project\venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Uploading receives data to BigQuery...
Raw staging tables successfully pushed to BigQuery!
